# 01 — Document Cleanup and OCR

**Question:** When does preprocessing make OCR easier, and when can it destroy useful character information?

Run every cell in order. Read the explanation first, predict the result, and then compare your prediction with the output.

## 1. Imports and paths

OpenCV stores color images as BGR rather than RGB. Matplotlib expects RGB, so we convert when displaying color images. All generated artifacts go into this project's own `output/` folder.

In [ ]:
from pathlib import Path
import shutil
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pytesseract

HERE = Path.cwd()
if not (HERE / 'data').exists():
    HERE = HERE / '01_document_cleanup_ocr'
DATA = HERE / 'data'
OUTPUT = HERE / 'output'
OUTPUT.mkdir(exist_ok=True)
print('Working directory:', HERE.resolve())
print('Tesseract found:', shutil.which('tesseract') is not None)

## 2. Load and inspect the pixels

An image is a NumPy array. Its shape is `(height, width, channels)`. Pixel values usually range from 0 (dark) to 255 (bright).

In [ ]:
image = cv2.imread(str(DATA / 'imageTextN.png'))
if image is None:
    raise FileNotFoundError('data/imageTextN.png was not found')
print('shape:', image.shape, 'dtype:', image.dtype, 'range:', (image.min(), image.max()))
plt.figure(figsize=(14, 6))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.title('Original document')
plt.axis('off');

## 3. Grayscale and denoising

Grayscale removes color dimensions that are not needed here. A bilateral filter smooths small intensity changes while trying to preserve edges. Strong filters can erase thin character strokes, so preprocessing always has a trade-off.

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
denoised = cv2.bilateralFilter(gray, d=7, sigmaColor=35, sigmaSpace=35)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for axis, stage, title in zip(axes, [gray, denoised], ['Grayscale', 'Denoised']):
    axis.imshow(stage, cmap='gray', vmin=0, vmax=255)
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout();

## 4. Compare global and local binarization

Otsu chooses one threshold for the whole page by minimizing within-class variance. Adaptive thresholding computes thresholds in local neighborhoods and is useful for uneven lighting.

In [ ]:
_, otsu = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
adaptive = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY, 31, 15)
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for axis, stage, title in zip(axes, [otsu, adaptive], ['Otsu', 'Adaptive']):
    axis.imshow(stage, cmap='gray', vmin=0, vmax=255)
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout();

## 5. Estimate and correct skew

We collect foreground pixel coordinates, fit the smallest rotated rectangle, and undo its angle. This simple method works for dense text but may be confused by photographs, borders, or multi-column layouts.

In [ ]:
def estimate_skew(binary):
    points = np.column_stack(np.where(binary < 128))
    if len(points) < 20:
        return 0.0
    angle = cv2.minAreaRect(points[:, ::-1].astype(np.float32))[-1]
    return -(90 + angle) if angle < -45 else -angle

def rotate_white(source, angle):
    height, width = source.shape[:2]
    matrix = cv2.getRotationMatrix2D((width / 2, height / 2), angle, 1.0)
    return cv2.warpAffine(source, matrix, (width, height), flags=cv2.INTER_CUBIC,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=255)

angle = estimate_skew(otsu)
deskewed = rotate_white(otsu, angle)
print(f'Applied deskew angle: {angle:.3f} degrees')
plt.figure(figsize=(14, 6))
plt.imshow(deskewed, cmap='gray')
plt.title('Final preprocessed image')
plt.axis('off');

## 6. Save stages and run OCR

Page segmentation mode 6 tells Tesseract to expect one uniform block of text. Try mode 3 for automatic layout analysis. If the next cell reports that Tesseract is missing, install the system package shown in the README and rerun it.

In [ ]:
stages = {'01_gray.png': gray, '02_denoised.png': denoised,
          '03_otsu.png': otsu, '04_adaptive.png': adaptive,
          '05_deskewed.png': deskewed}
for filename, stage in stages.items():
    cv2.imwrite(str(OUTPUT / filename), stage)

if shutil.which('tesseract') is None:
    raise RuntimeError('Install Tesseract first: brew install tesseract')
raw_text = pytesseract.image_to_string(image, config='--psm 6').strip()
clean_text = pytesseract.image_to_string(deskewed, config='--psm 6').strip()
(OUTPUT / 'raw_ocr.txt').write_text(raw_text, encoding='utf-8')
(OUTPUT / 'preprocessed_ocr.txt').write_text(clean_text, encoding='utf-8')
print('RAW OCR:')
print(raw_text)
print()
print('PREPROCESSED OCR:')
print(clean_text)

## Reflection

1. Which visible character details changed after thresholding?
2. Did cleaner-looking pixels produce better text?
3. Which preprocessing decision would you validate on more pages before deployment?

Continue with project 2 to replace visual judgment with CER and WER.